# Цель данного документа

# Подготовка

## Подключение Google Drive

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Установка пакетов

In [2]:
! pip install -r "/content/drive/MyDrive/Colab Notebooks/datapipe_demo_1/requirements.txt"

Ignoring colorama: markers 'python_version == "3.12" and platform_system == "Windows"' don't match your environment
Ignoring tzdata: markers '(sys_platform == "win32" or sys_platform == "emscripten") and python_version == "3.12"' don't match your environment


## Импорты

In [3]:
from sqlalchemy import Column, String, Integer

from datapipe.compute import Catalog
from datapipe.compute import DatapipeApp
from datapipe.compute import Pipeline
from datapipe.compute import Table
from datapipe.executor import ExecutorConfig
from datapipe.datatable import DataStore
from datapipe.step.batch_transform import BatchTransform
from datapipe.store.database import DBConn
from datapipe.store.pandas import TableStoreJsonLine

import os
import re
import time
from pathlib import Path
from typing import List, Optional

import fsspec
import pandas as pd
from datapipe.compute import ComputeStep, PipelineStep
from datapipe.step.datatable_transform import DatatableTransformStep
from datapipe.datatable import DataTable
from datapipe.run_config import RunConfig
from datapipe.store.filedir import (
    _pattern_to_attrnames,
    _pattern_to_glob,
    _pattern_to_match,
    _pattern_to_patterns_or,
)
from datapipe.types import Labels

## Функция сканирования входных данных

In [4]:
FILES_SCHEMA = [
    Column("filepath", String()),
    Column("size_bytes", Integer()),
]


class ScanFileList(PipelineStep):
    def __init__(
        self,
        filename_pattern: str,
        filename_output: str,
        output: str,
        labels: Optional[Labels] = None,
    ):
        self.output = output
        self.filename_pattern = filename_pattern
        self.filename_output = filename_output
        self.labels = labels

    def build_compute(self, ds: DataStore, catalog: Catalog) -> List[ComputeStep]:
        attrnames = _pattern_to_attrnames(self.filename_pattern)

        catalog.add_datatable(
            name=self.output,
            dt=Table(
                store=TableStoreJsonLine(
                    filename=self.filename_output.format(data=self.output),
                    primary_schema=[
                        Column(attrname, String(), primary_key=True)
                        for attrname in attrnames
                    ]
                    + FILES_SCHEMA,
                )
            ),
        )

        output_table = catalog.get_datatable(ds, self.output)

        return [
            DatatableTransformStep(
                name=f"scan_file_list__{self.output}",
                input_dts=[],
                output_dts=[output_table],
                func=scan_file_list,
                kwargs={"filename_pattern": self.filename_pattern},
                labels=self.labels,
            )
        ]


def scan_file_list(
    ds: DataStore,
    input_dts: List[DataTable],
    output_dts: List[DataTable],
    run_config: Optional[RunConfig],
    kwargs: dict,
) -> None:
    [output_dt] = output_dts
    now = time.time()

    filename_pattern = kwargs["filename_pattern"]

    protocol, path = fsspec.core.split_protocol(filename_pattern)

    if protocol is None or protocol == "file":
        filename_pattern = str(Path(path).resolve())
        filename_pattern_for_match = filename_pattern
        protocol_str = "" if protocol is None else "file://"
    else:
        filename_pattern = str(filename_pattern)
        filename_pattern_for_match = path
        protocol_str = f"{protocol}://"

    filename_patterns = _pattern_to_patterns_or(filename_pattern)
    attrnames = _pattern_to_attrnames(filename_pattern)
    filename_glob = [_pattern_to_glob(pat) for pat in filename_patterns]
    filename_match = _pattern_to_match(filename_pattern_for_match)

    files = fsspec.open_files(filename_glob)

    res = []

    for f in files:
        item = {
            "filepath": f"{protocol_str}{os.path.relpath(f.path)}",
            "size_bytes": files.fs.size(f.path),
        }

        m = re.match(filename_match, f.path)

        assert m is not None
        for attrname in attrnames:
            item[attrname] = m.group(attrname)

        res.append(item)

    output_dt.store_chunk(pd.DataFrame(res), now=now, run_config=run_config)
    output_dt.delete_stale_by_process_ts(now, now=now, run_config=run_config)

## Создание SQLite БД для меты

In [5]:
try:
    import pysqlite3
    sqla_engine = "sqlite+pysqlite3"
except ImportError:
    sqla_engine = "sqlite"

db_path_str = "/content/drive/MyDrive/Colab Notebooks/datapipe_demo_1/store.sqlite"
db_dir = Path(db_path_str).parent

# Ensure the directory exists before trying to create the database file
os.makedirs(db_dir, exist_ok=True)

dbconn = DBConn(f"{sqla_engine}:///{db_path_str}")
ds = DataStore(dbconn)

/usr/local/lib/python3.12/dist-packages/datapipe/store/database.py:65: SAWarning: Dialect sqlite:pysqlite will not make use of SQL compilation caching as it does not set the 'supports_statement_cache' attribute to ``True``.  This can have significant performance implications including some performance degradations in comparison to prior SQLAlchemy versions.  Dialect maintainers should seek to set this attribute to True after appropriate development and testing for SQLAlchemy 1.4 caching support.   Alternatively, this attribute may be set to False which will disable this warning. (Background on this warning at: https://sqlalche.me/e/20/cprf)
  con.execute(text("PRAGMA journal_mode=WAL"))


# Шаг 1. Сканирование данных

In [6]:
FILEPATH__RAW__CARS = "/content/drive/MyDrive/Colab Notebooks/datapipe_demo_1/data/raw/cars/{file_name}.json"
FILEPATH__PROCESSED__CARS = "/content/drive/MyDrive/Colab Notebooks/datapipe_demo_1/data/processed/cars/{data}.jsonl"

In [7]:
catalog = Catalog({})

In [8]:
pipeline = Pipeline(
    [
        ScanFileList(
            filename_pattern=FILEPATH__RAW__CARS,
            filename_output=FILEPATH__PROCESSED__CARS,
            output="cars_scanned",
            labels=[
                ("entity", "cars"),
                ("layer", "scan"),
                ("environment", "prod"),
            ],
        ),
    ]
)

In [9]:
app = DatapipeApp(ds, catalog, pipeline)

In [10]:
app.ds.meta_dbconn.sqla_metadata.create_all(app.ds.meta_dbconn.con)

In [11]:
from datapipe.compute import run_steps

run_steps(app.ds, app.steps)

In [15]:
with open(FILEPATH__PROCESSED__CARS.replace("{data}", "cars_scanned"), "r") as f:
  for line in f.readlines():
    print(line)

{"file_name":"record_1","filepath":"drive\/MyDrive\/Colab Notebooks\/datapipe_demo_1\/data\/raw\/cars\/record_1.json","size_bytes":136}

{"file_name":"record_2","filepath":"drive\/MyDrive\/Colab Notebooks\/datapipe_demo_1\/data\/raw\/cars\/record_2.json","size_bytes":136}

{"file_name":"record_3","filepath":"drive\/MyDrive\/Colab Notebooks\/datapipe_demo_1\/data\/raw\/cars\/record_3.json","size_bytes":136}

{"file_name":"record_4","filepath":"drive\/MyDrive\/Colab Notebooks\/datapipe_demo_1\/data\/raw\/cars\/record_4.json","size_bytes":134}

{"file_name":"record_5","filepath":"drive\/MyDrive\/Colab Notebooks\/datapipe_demo_1\/data\/raw\/cars\/record_5.json","size_bytes":134}

